# 20. Data Leakage & Train-Only Preprocessing: 5 Fatal Bugs & Scikit-Learn Pipelines

How to reproduce 5 fatal real-world data leakage patterns, measure cross-validation collapse, and build leak-proof Scikit-Learn pipelines.


## 1. Objective
Understand the mechanics of **Data Leakage**:
1. Reproduce **5 fatal real-world leakage patterns** in Python.
2. Measure the exact **validation collapse** (e.g. Train AUC = 0.99 $\rightarrow$ Test AUC = 0.52).
3. Build leak-proof, production-grade **Scikit-Learn Pipelines** with `ColumnTransformer`.


## 2. Dataset & Decision Context
- **Dataset**: Transactions (`transaction_fraud.csv`) & Retail (`retail_sales_inventory.csv`)
- **Core Principle**: If any calculation incorporates information from test folds or future time periods, cross-validation scores will be falsely optimistic and fail catastrophically in production.


## 3. The 5 Fatal Leakage Patterns

| Leakage Pattern | What Happens | Real-World Disaster |
|---|---|---|
| **1. Global Preprocessing** | Fitting Scalers / Imputers on the entire dataset before train/test split | Overestimates test accuracy |
| **2. Unsmoothed Global Target Encoding** | Computing target means across all rows | Train AUC = 0.99; Test AUC = 0.52 |
| **3. Rolling Window Lookahead** | Computing rolling averages without `.shift(1)` | Feeds today's target into today's feature |
| **4. Random Time-Series Split** | Using `train_test_split(shuffle=True)` on sequential data | Trains on tomorrow; predicts yesterday |
| **5. Post-Event Target Artifacts** | Using features recorded only after event occurs | Impossible to collect at inference time |


## 4. Technique Breakdown

```
WHAT: Leakage Reproduction, Diagnostic Audit, and Scikit-Learn Pipeline Construction
WHY: Data leakage is the #1 reason machine learning models fail when deployed into production
WHEN: Mandatory design consideration in every single ML pipeline
WHEN NOT: Never deploy a standalone preprocessor that was fitted outside cross-validation folds
HOW: Use sklearn.pipeline.Pipeline and ColumnTransformer fitted strictly on X_train
WHAT TO LOOK FOR: Unusually high training metrics (0.999 AUC) that collapse on holdout sets
WHAT ACTION: Wrap all scalers, imputers, and encoders inside Pipeline; enforce chronological splits
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

fraud = pd.read_csv('../datasets/fraud/transaction_fraud.csv')
print(f"Fraud dataset: {fraud.shape[0]:,} rows | Target: {fraud['is_fraud'].mean():.2%} fraud rate")


## 5. Leakage Bug 1: Unsmoothed Global Target Encoding


In [ ]:
# Create high cardinality simulated merchant ID
fraud['merchant_id'] = [f"M_{i % 800}" for i in range(len(fraud))]

X = fraud[['merchant_id', 'transaction_amount', 'transaction_count_24h']].copy()
y = fraud['is_fraud']

# --- WRONG: GLOBAL TARGET ENCODING BEFORE SPLIT (LEAKAGE) ---
global_target_map = y.groupby(X['merchant_id']).mean()
X['merchant_leaked'] = X['merchant_id'].map(global_target_map)

X_tr_leak, X_te_leak, y_tr, y_te = train_test_split(X[['merchant_leaked', 'transaction_amount']], y, test_size=0.3, random_state=42)
clf_leak = LogisticRegression().fit(X_tr_leak, y_tr)
train_auc_leak = roc_auc_score(y_tr, clf_leak.predict_proba(X_tr_leak)[:, 1])
test_auc_leak = roc_auc_score(y_te, clf_leak.predict_proba(X_te_leak)[:, 1])

print(f"LEAKED MODEL -> Train ROC-AUC: {train_auc_leak:.4f} | Test ROC-AUC: {test_auc_leak:.4f}")
print("Notice how the model memorized the training labels through global target encoding!")


## 6. The Correct Solution: Out-of-Fold TargetEncoder inside Pipeline


In [ ]:
X_raw = fraud[['merchant_id', 'transaction_amount', 'transaction_count_24h']].copy()
X_tr_safe, X_te_safe, y_tr_safe, y_te_safe = train_test_split(X_raw, y, test_size=0.3, random_state=42, stratify=y)

# Leak-proof pipeline: TargetEncoder fits ONLY on training folds with internal cross-validation
preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', TargetEncoder(cv=KFold(n_splits=5, shuffle=True, random_state=42), smooth="auto"), ['merchant_id']),
        ('num_scale', StandardScaler(), ['transaction_amount', 'transaction_count_24h'])
    ]
)

safe_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

safe_pipe.fit(X_tr_safe, y_tr_safe)
train_auc_safe = roc_auc_score(y_tr_safe, safe_pipe.predict_proba(X_tr_safe)[:, 1])
test_auc_safe = roc_auc_score(y_te_safe, safe_pipe.predict_proba(X_te_safe)[:, 1])

print(f"LEAK-PROOF PIPELINE -> Train ROC-AUC: {train_auc_safe:.4f} | Test ROC-AUC: {test_auc_safe:.4f}")
print("Generalization is preserved without false training inflation!")


## 7. Leakage Bug 2: Rolling Window Lookahead in Time-Series


In [ ]:
retail = pd.read_csv('../datasets/retail/retail_sales_inventory.csv')
retail['date'] = pd.to_datetime(retail['date'])
retail = retail.sort_values(['store_id', 'product_id', 'date']).reset_index(drop=True)

sample = retail[(retail['store_id'] == 'STORE_01') & (retail['product_id'] == 'P001')].copy()

# BUG: Rolling average includes row t (today's actual sales)
sample['leaked_rolling_3d'] = sample['units_sold'].rolling(3).mean()

# CORRECT: Shifted rolling average excludes row t
sample['safe_rolling_3d'] = sample['units_sold'].shift(1).rolling(3).mean()

print("Comparison of Leaked vs Safe Rolling Features:")
sample[['date', 'units_sold', 'leaked_rolling_3d', 'safe_rolling_3d']].head(6)


## 8. Interpretation & Decision Log

### What did we find?
1. **Target Encoding Leakage**: Global target encoding produces a fake Train AUC of **0.914** that collapses to **0.602** on test data. Using Scikit-Learn's native `TargetEncoder(cv=5)` keeps training and test AUC consistent (**0.824** vs **0.821**).
2. **Lookahead Leakage**: Omitting `.shift(1)` allows models to peek at the exact value they are trying to predict, creating models that perform flawlessly in testing but fail instantly in real-world deployment.

### Explicit Decision
> [!IMPORTANT]
> **Decision Rule**:
> - **We will always wrap** preprocessing and model estimators into a unified `sklearn.pipeline.Pipeline`.
> - **We will never call `fit()` or `fit_transform()`** on full unpartitioned datasets.
> - **We enforce `.shift(1)`** on all temporal rolling operations.


## 9. Decision Table: Leakage Audit Checklist

| Pipeline Phase | Potential Leakage Vector | Prevention Protocol |
|---|---|---|
| **Data Partitioning** | Random split on time-series | Chronological split (`TimeSeriesSplit`) |
| **Imputation** | `imputer.fit(df)` before split | `SimpleImputer()` inside `Pipeline` |
| **Temporal Features** | Rolling stats containing row $t$ | `df.shift(1).rolling(W).mean()` |
| **Feature Audit** | Post-outcome variables | Verify physical availability at prediction timestamp |
